# 00 Report files

The analysis compares the calls with 29 UBS filings: eight quarterly reports and six Pillar 3 reports for 2023 and 2024, the annual reports 2023 and 2024, and the 13 reports of 2025 and 2026. This notebook fetches the filings with a recorded source from EDGAR into `data/reports/` and compares their hashes with the manifest. It is optional: the other notebooks run without the reports and then skip the verification of quotes.

In [1]:
# project setup - run first, do not edit except NAME
NAME = "call_only_targets"

import sys, subprocess, pathlib
IN_COLAB = "google.colab" in sys.modules
REPO = "boe-financial-analysis"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    if not pathlib.Path(f"/content/{REPO}").exists():
        subprocess.run(["git", "clone", "-q",
                        f"https://github.com/maybebool/{REPO}.git",
                        f"/content/{REPO}"], check=True)
    ROOT = pathlib.Path(f"/content/{REPO}")
    DATA = pathlib.Path("/content/drive/MyDrive/boe-data")
    subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin"], check=False)
    subprocess.run(["git", "-C", str(ROOT), "merge", "-q", "origin/main", "-m", "sync"],
                   check=False)
else:
    ROOT = pathlib.Path.cwd()
    while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    DATA = ROOT / "data"

sys.path.insert(0, str(ROOT))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(ROOT / "requirements.txt")], check=True)

from notebooks.env_cell import verify, check_python
check_python()
try:
    verify(ROOT)
except RuntimeError as e:
    print(e)
    print("\n Runtime -> Restart session, then run this cell again.")
    raise

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src import loading

print(f"ok  python {sys.version.split()[0]}  pandas {pd.__version__}")

ok  python 3.13.15  pandas 2.2.3


In [2]:
# folders and helpers shared by the notebooks of this section
import json, re
NB_DIR = ROOT / "notebooks" / "05_call_only_targets"
sys.path.insert(0, str(NB_DIR))
import report_checks as rc

IN = DATA / "call_only_targets"        # delivered result files
REPORTS = DATA / "reports"             # report files, optional (see 00_get_reports)
EXPORTS = NB_DIR / "exports"
EXPORTS.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", 160, "display.max_rows", 60)

manifest = pd.read_csv(IN / "reports_manifest.csv", keep_default_na=False)
status = rc.report_status(REPORTS, manifest)
print(rc.mode_message(status))

REPORTS PRESENT (29 of 29): every quote is verified against the text of its page.


## Manifest

In [3]:
view = manifest.merge(status, on="file")
view["source recorded"] = view.source_url != ""
view[["file", "bytes", "source recorded", "present", "sha256_matches"]]

,file,bytes,source recorded,present,sha256_matches
0,UBS_2023-Q1_report.htm,4590066,False,True,True
1,UBS_2023-Q2_report.htm,8941344,False,True,True
2,UBS_2023-Q3_pillar3.htm,1558948,False,True,True
3,UBS_2023-Q3_report.htm,4495971,False,True,True
4,UBS_2023-Q4_pillar3.htm,5320943,False,True,True
5,UBS_2023-Q4_report.htm,2653174,False,True,True
6,UBS_2023_annual_report.htm,21589503,False,True,True
7,UBS_2024-Q1_pillar3.htm,1539895,False,True,True
8,UBS_2024-Q1_report.htm,3804162,False,True,True
9,UBS_2024-Q2_pillar3.htm,2886741,False,True,True


In [4]:
print("reports in the manifest:", len(view))
print("with a recorded source:", int(view["source recorded"].sum()))
print("present locally:", int(view.present.sum()), "| hash equal to the manifest:", int(view.sha256_matches.sum()))

reports in the manifest: 29
with a recorded source: 15
present locally: 29 | hash equal to the manifest: 29


The manifest lists the SHA-256 of every file used in the analysis and, after removal of the script tag that EDGAR adds on delivery, a normalised hash that is stable across downloads. A source address is recorded for 15 of the 29 files.

## Download

EDGAR requires a declared user agent. Set the environment variable `SEC_USER_AGENT` to a name and a contact address before running the next cell; without it nothing is downloaded. Files that are already present are not fetched again.

In [5]:
import os, time, urllib.error, urllib.request

PAUSE_SECONDS = 1.0                       # well below the EDGAR limit of ten requests per second
agent = os.environ.get("SEC_USER_AGENT", "").strip()
todo = view[(view.source_url != "") & ~view.present]
if todo.empty:
    print("nothing to download: every file with a recorded source is present")
elif not agent:
    print(f"SEC_USER_AGENT is not set: {len(todo)} file(s) not downloaded")
else:
    REPORTS.mkdir(parents=True, exist_ok=True)
    for r in todo.itertuples():
        req = urllib.request.Request(r.source_url, headers={"User-Agent": agent})
        try:
            with urllib.request.urlopen(req, timeout=120) as resp:
                raw = resp.read()
        except urllib.error.HTTPError as err:      # EDGAR refuses undeclared or blocked agents with status 403
            print(f"{r.file}: not downloaded, HTTP status {err.code}")
            time.sleep(PAUSE_SECONDS)
            continue
        (REPORTS / r.file).write_bytes(raw)
        same = rc.normalized_sha256(raw) == r.sha256_normalized
        print(f"{r.file}: {len(raw):,} bytes, normalised hash {'equal to' if same else 'DIFFERENT from'} the manifest")
        time.sleep(PAUSE_SECONDS)

nothing to download: every file with a recorded source is present


## Files to obtain by hand

In [6]:
status = rc.report_status(REPORTS, manifest)
view = manifest.merge(status, on="file")
by_hand = view[(view.source_url == "") & ~view.present][["file", "sha256", "bytes"]]
print("files without a recorded source that are not present:", len(by_hand))
by_hand

files without a recorded source that are not present: 0


,file,sha256,bytes


In [7]:
no_source = view[view.source_url == ""][["file", "sha256", "present", "sha256_matches"]]
print("files without a recorded source:", len(no_source))
no_source

files without a recorded source: 14


,file,sha256,present,sha256_matches
0,UBS_2023-Q1_report.htm,3c7252605a1fbb08315ba50c713ae2f68d8754a438f991ec79037df33045cf34,True,True
1,UBS_2023-Q2_report.htm,2b1842c28d6c24a5565448ec26332c778e1fe8a168f3bbade005f83b1158ba36,True,True
2,UBS_2023-Q3_pillar3.htm,1e8e7df0191dc654817f61a30cb38e1016729862b066c14993edd4b07fe6c7ae,True,True
3,UBS_2023-Q3_report.htm,0c1fd4f1ecf438c058d17253c977339126beca53d46a2fb37d8108b007bfb505,True,True
4,UBS_2023-Q4_pillar3.htm,c0466863017c7394a1b750b205ff599db2ddb7b89af6810218928aff327bbd59,True,True
5,UBS_2023-Q4_report.htm,be84ee7e16ed6b9db1466098789b61a44c0956a442b9eca906d2347b757ce440,True,True
6,UBS_2023_annual_report.htm,69b271b74a77ea27997d80e3822d571b520e9d7536e32375d3e6dade0b9b820e,True,True
7,UBS_2024-Q1_pillar3.htm,93f9b1e050dc70ff631bc6c08f70cb3e08faf6ec93901c348336c398ac4e4e60,True,True
8,UBS_2024-Q1_report.htm,cab7aa3010367976768c501b9669827ebffb61b33ea48c670d1643abd3d5fd9a,True,True
9,UBS_2024-Q2_pillar3.htm,82bb08ffc5cefd43eda78d90209c2248289b531d0ee75d9103912fb6bb321986,True,True


For 14 filings (the quarterly reports 2023-Q1 to 2024-Q3, the annual report 2023 and the six Pillar 3 reports of 2023 and 2024) no source address is recorded. They are the corresponding Form 6-K and Form 20-F filings of UBS Group AG on EDGAR; a copy placed in `data/reports/` under the listed name is accepted when its SHA-256 equals the manifest.

In [8]:
print(rc.mode_message(status))
assert len(manifest) == 29 and manifest.file.is_unique
assert (status.sha256_matches | ~status.present).all(), "a present file differs from the manifest"

REPORTS PRESENT (29 of 29): every quote is verified against the text of its page.
